In [4]:
import sounddevice as sd
import torch
import soundfile as sf
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor, AutoProcessor, AutoModelForCTC


processor = AutoProcessor.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn")
model = AutoModelForCTC.from_pretrained("jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn")

# 设置录音参数
samplerate = 16000
duration = 10  # 秒

def record_audio(duration, samplerate):
    print("Recording...")
    audio = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()  # 等待录音结束
    print("Recording complete.")
    return audio.flatten()

def transcribe(audio):
    inputs = processor(audio, sampling_rate=samplerate, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(inputs.input_values).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = processor.batch_decode(predicted_ids)[0]
    return transcription

# 录音
audio = record_audio(duration, samplerate)

# 语音识别
transcription = transcribe(audio)
print("Transcription: ", transcription)


Some weights of the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn were not used when initializing Wav2Vec2ForCTC: ['wav2vec2.encoder.pos_conv_embed.conv.weight_g', 'wav2vec2.encoder.pos_conv_embed.conv.weight_v']
- This IS expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Wav2Vec2ForCTC from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at jonatasgrosman/wav2vec2-large-xlsr-53-chinese-zh-cn and are newly initialized: ['wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.original0', 'wav2vec2.encoder.pos_conv_embed.conv.parametrizations.weight.o

Recording...
Recording complete.
Transcription:  我今天中五应该吃什么


In [5]:
import os
import openai

# 设置 OpenAI API 密钥
api_key = ''

def generate_response(transcription):
    client = openai.OpenAI(api_key=api_key)
    
    response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "你好，请注意你现在生成的文字要按照人日常生活的口吻，你的回复将会后续用TTS模型转为语音，并且请把回答控制在100字以内。将数字等转为文字回答。"},
        {"role": "user", "content": f"这是一个老人，所以说话可能有些歧义，请根据以下内容提供相应的服务，语气要温和一点，尽量给出回答。文本：{transcription}"}
    ],
    max_tokens=256,
    temperature=0.5
)


    return response.choices[0].message.content.strip()


# 生成回复
response = generate_response(transcription)
print("Response: ", response)

Response:  您好！中午应该吃些清淡易消化的食物，比如蔬菜水果、米饭面条、清汤等。可以搭配一些肉类或豆制品，保持营养均衡。记得多喝水，注意饮食健康哦！祝您用餐愉快！


In [8]:
import torch
torch._dynamo.config.cache_size_limit = 64
torch._dynamo.config.suppress_errors = True
torch.set_float32_matmul_precision('high')
import soundfile as sf

import ChatTTS.ChatTTS as ChatTTS
from IPython.display import Audio

chat = ChatTTS.Chat()
chat.load_models()

params_infer_code = {'prompt':'[speed_1]', 'temperature':.3}
params_refine_text = {'prompt':'[oral_0][laugh_0][break_10]'}

wavs = chat.infer(response)
sf.write('output.wav', wavs[0][0], 24000)

Audio(wavs[0], rate=24_000, autoplay=True)


INFO:ChatTTS.ChatTTS.core:Load from cache: C:\Users\HenryLi/.cache/huggingface\hub/models--2Noise--ChatTTS/snapshots\cc14302f34d7855eb3420d1fd48345012ff1460d
INFO:ChatTTS.ChatTTS.core:use cuda:0
INFO:ChatTTS.ChatTTS.core:vocos loaded.
INFO:ChatTTS.ChatTTS.core:dvae loaded.
INFO:ChatTTS.ChatTTS.core:gpt loaded.
INFO:ChatTTS.ChatTTS.core:decoder loaded.
INFO:ChatTTS.ChatTTS.core:tokenizer loaded.
INFO:ChatTTS.ChatTTS.core:All initialized.
INFO:ChatTTS.ChatTTS.core:All initialized.
2024-06-18 11:28:47,401 WETEXT INFO found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
2024-06-18 11:28:47,401 WETEXT INFO found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
2024-06-18 11:28:47,401 WETEXT INFO found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
INFO:wetext-zh_normalizer:found existing fst: c:\Users\HenryLi\.conda\envs\VUI\lib\site-packages\tn\zh_tn_tagger.fst
2024-06-18 11:28:4

[[-1.5328795e-05 -4.0705265e-05 -4.0032264e-05 ... -5.6204271e-06
   5.1187103e-06  1.1142875e-05]]
